# Build, Shape, and Extend an OpenClaw Agent on AMD

---

**What you have in front of you:**
- This notebook — the workshop guide. Read it, run the occasional cell.
- A terminal on the side — where the real interaction happens via the OpenClaw TUI.

**Format:**
- ▶ **Run this cell** → execute it in the notebook (Shift+Enter)
- 🖥️ **In the terminal / TUI** → go to your terminal and run this
- 👀 **Behind the scenes** → we peek at what the agent actually wrote to disk

---

## What You'll Learn Today

You're running on an **AMD MI300X** — one of the most powerful AI accelerators available. By the end of this workshop you'll have used it to run a 122-billion-parameter model locally and built a fully autonomous agent on top of it.

No cloud. No API keys. No rate limits. Everything runs on the hardware in front of you.

---

### The Arc

**Part 1 — Run the model**
Serve a state-of-the-art open-weight LLM locally using SGLang on AMD hardware. Understand what's happening under the hood when the model is ready.

**Part 2 — Meet your agent**
Start OpenClaw and go through onboarding. See exactly what the agent writes to disk when it learns who you are — and understand the file-based memory system that makes it persistent across sessions.

**Part 3 — Shape its behaviour**
Agents aren't fixed. You'll edit a plain markdown file to layer new behavioural rules on top of the agent's personality — and immediately see the effect on how it explains things.

**Part 4 — Give it something to do**
The agent clones a real Python project, runs the test suite, finds a bug, fixes it, and verifies the fix — all autonomously. You'll watch the think → act → observe → repeat loop in real time.

**Part 5 — Package the behaviour as a skill**
Turn what the agent just did into a reusable skill: a markdown file that makes the same debugging workflow available on any Python project, forever.

**Part 6 — Make it work for you, every morning**
Tell the agent — in plain English — what you want to wake up to. It will schedule itself, save your preferences to memory, fetch live data from GitHub and the web, and deliver a personalised brief. Then watch it transition from reporting to actively fixing code.

**Part 7 — The challenge**
Apply everything you've built to an open-ended problem. Three tracks to choose from — pick the one that interests you most and see how far you can push the agent.

---

### What You'll Walk Away With

| | |
|---|---|
| A running local LLM stack | SGLang + AMD, fully configured |
| A personalised agent | Knows your name, preferences, and working style |
| A reusable skill | Works on any Python repo, not just today's |
| An autonomous workflow | Runs every morning without you asking |
| A challenge result | Something to demo, share, or build on |

---

*Let's start.*

---
## Section 0 — Setup

Run these four cells once. The first starts the model server on AMD hardware, the remaining three verify it's up, connect OpenClaw to it, and start the gateway.

### 🖥️ Step 1 — Start the model server

The cell below starts SGLang inside the container to serve **Qwen3.5-122B** on AMD hardware. The first run downloads the model weights — this can take a few minutes. Subsequent runs start in seconds.

> **Note:** The API key `abc-123` is used throughout this notebook. You can replace it with any string — just keep it consistent across all cells.

In [ ]:
import subprocess, time, pathlib

result = subprocess.run([
    "docker", "run", "-d",
    "--name", "sglang_server",
    "--ipc=host",
    "--privileged",
    "--device=/dev/kfd",
    "--device=/dev/dri",
    "-p", "8090:8090",
    "lmsysorg/sglang:v0.5.9-rocm700-mi30x",
    "python3", "-m", "sglang.launch_server",
    "--model-path", "Qwen/Qwen3.5-122B-A10B-FP8",
    "--served-model-name", "qwen3-5-122b",
    "--host", "0.0.0.0",
    "--port", "8090",
    "--tp-size", "1",
    "--api-key", "abc-123",
    "--attention-backend", "triton",
    "--reasoning-parser", "qwen3",
    "--tool-call-parser", "qwen3_coder",
    "--trust-remote-code",
], capture_output=True, text=True)

if result.returncode == 0:
    container_id = result.stdout.strip()[:12]
    print(f"✅ Container started ({container_id})")
    print("   Model loading — this takes a few minutes on first run.")
    print("   Run the next cell once to verify SGLang is ready before continuing.")
elif "already in use" in result.stderr:
    print("ℹ️  Container already running — nothing to do.")
else:
    print("❌ Docker error:")
    print(result.stderr[-400:])

In [ ]:
import urllib.request, json, subprocess, time, pathlib, sys

# Step 2 — Wait for SGLang to be ready (polls up to 5 minutes)
print("Waiting for SGLang to be ready", end="", flush=True)
deadline = time.time() + 300
ready = False
while time.time() < deadline:
    try:
        req = urllib.request.Request(
            "http://localhost:8090/v1/models",
            headers={"Authorization": "Bearer abc-123"}
        )
        with urllib.request.urlopen(req, timeout=3) as r:
            models = json.loads(r.read())
        ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)

if ready:
    print("\n✅ SGLang is ready")
    for m in models.get("data", []):
        print(f"   Model: {m['id']}")
else:
    print("\n❌ SGLang did not become ready within 5 minutes — check docker logs sglang_server")

In [ ]:
BASE_URL = "http://localhost:8090/v1"
API_KEY  = "abc-123"
MODEL_ID = "qwen3-5-122b"

subprocess.run(["openclaw", "config", "set", "gateway.mode", "local"], check=True)
subprocess.run(["openclaw", "config", "set", "agents.defaults.model", f"sglang/{MODEL_ID}"], check=True)

config_path = pathlib.Path.home() / ".openclaw" / "openclaw.json"
with config_path.open() as f:
    cfg = json.load(f)

cfg.setdefault("models", {}).setdefault("providers", {})["sglang"] = {
    "baseUrl": BASE_URL, "apiKey": API_KEY, "api": "openai-completions",
    "models": [{"id": MODEL_ID, "name": MODEL_ID, "reasoning": True,
                "input": ["text"], "cost": {"input": 0, "output": 0, "cacheRead": 0, "cacheWrite": 0},
                "contextWindow": 131072, "maxTokens": 8192}]
}
with config_path.open("w") as f:
    json.dump(cfg, f, indent=2)
print("✅ OpenClaw configured → Qwen3.5-122B on SGLang")

In [ ]:
log_file = pathlib.Path("/tmp/openclaw-gateway.log")
subprocess.run(["pkill", "-9", "-f", "openclaw-gateway"], capture_output=True)
time.sleep(1)
proc = subprocess.Popen(
    ["openclaw", "gateway", "run", "--bind", "loopback", "--port", "18789", "--force"],
    stdout=log_file.open("w"), stderr=subprocess.STDOUT,
)
time.sleep(4)
if proc.poll() is None:
    print(f"✅ Gateway running (PID {proc.pid})")
else:
    print("❌ Gateway failed:\n", log_file.read_text()[-600:])

---
## The Map

Before we start, here's everything you'll touch today and what it does:

| Concept | What it is | Where it lives |
|---|---|---|
| **Gateway** | Connects OpenClaw to the model running on AMD hardware | Process on port 18789 |
| **Workspace** | The agent's "brain" — files it reads on every message | `~/.openclaw/workspace/` |
| **Tools** | How the agent acts on the world: read files, run shell commands, write edits | Defined in `TOOLS.md` |
| **Skills** | Saved, reusable workflows the agent can follow on demand | `workspace/skills/` |

The key insight: **there is no magic**. The agent's memory, personality, and skills are plain markdown files on disk. You can read them, edit them, and version-control them. That's what this workshop demonstrates.

---
## Section 1 — Onboarding

### 🖥️ In the TUI — start OpenClaw

```bash
openclaw
```

OpenClaw will ask you a few questions — your name, how you like to work, what kind of tone you prefer. Answer however you like.

When you're done, come back here.

### 👀 Behind the scenes — what just happened

Those answers didn't just disappear. The agent wrote them to files in its **workspace** — a folder it reads on every message to remember who you are and how to behave.

Run the next two cells to see exactly what was written.

In [ ]:
workspace = pathlib.Path.home() / ".openclaw" / "workspace"

print("=" * 60)
print("IDENTITY.md")
print("=" * 60)
print((workspace / "IDENTITY.md").read_text())

In [ ]:
print("=" * 60)
print("SOUL.md")
print("=" * 60)
print((workspace / "SOUL.md").read_text())

The workspace has 8 files that get injected into every session:

```
~/.openclaw/workspace/
├── SOUL.md       ← WHO the agent is: values, personality, tone
├── AGENTS.md     ← HOW it operates: startup rules, memory, red lines
├── IDENTITY.md   ← WHAT others see: name, emoji, public-facing metadata
├── USER.md       ← WHO you are: context the agent reads about you
├── TOOLS.md      ← local environment notes (SSH hosts, device names, etc.)
├── MEMORY.md     ← long-term curated memory across sessions
├── HEARTBEAT.md  ← checklist for periodic background checks
├── BOOTSTRAP.md  ← first-run ritual (deleted after onboarding)
└── memory/       ← daily session logs (today + yesterday auto-loaded)
```

**SOUL.md is personality. AGENTS.md is policy.** Every message the agent receives, it re-reads both. The startup sequence is literally written into `AGENTS.md`:

> *Before doing anything else: read SOUL.md, read USER.md, read today's memory log.*

Run the next cell to see AGENTS.md:"

In [ ]:
print("=" * 60)
print("AGENTS.md  (truncated)")
print("=" * 60)
agents_text = (workspace / "AGENTS.md").read_text()
print(agents_text[:1200] + "\n\n... (truncated)")

### Prove it — layer a rule into AGENTS.md before the bug fix

`SOUL.md` is personality — don't touch that. `AGENTS.md` is policy — that's where we add a rule about *how* to behave when debugging.

The next cell appends one section to `AGENTS.md`. After this, every debug explanation from the agent will follow those rules. Run it and see the before/after.

In [ ]:
debug_rule = """
---

## Debugging Policy (added during workshop)

When investigating and explaining bugs:
- State the bug in two sentences max: file, line, what it does wrong
- No filler phrases — just the root cause and the fix
- Use bullet points, never paragraphs
- After fixing, report exactly: file changed, line changed, before, after

"""

agents_path = workspace / "AGENTS.md"
original = agents_path.read_text()

print("── AGENTS.md BEFORE (last 200 chars) ──────────────────────")
print("..." + original[-200:])

agents_path.write_text(original + debug_rule)

print("\n── APPENDED ────────────────────────────────────────────────")
print(debug_rule)

The agent that does the bug fix in Section 4 will read the full file — original soul plus this new section. Watch how it explains the root cause compared to a plain `openclaw chat` with no shaping.

This is the pattern for real use too: you don't replace the soul for every task, you layer specialised instructions on top of a stable base.

---
## Section 2 — Pull the App

Now let's give the agent something to work with. Ask it to clone the typing app and set it up.

### 🖥️ In the TUI

```
Clone https://github.com/Mahdi-CV/open_type_faster and install its dependencies
```

Watch it use shell tools to clone, inspect the requirements, and install. This is the **act** phase — the agent doesn't just answer, it does.

---
## Section 3 — Run the App and Spot the Bug

### 🖥️ Open a new terminal and run the app yourself

```bash
cd ~/.openclaw/workspace/open_type_faster && ./venv/bin/python main.py
```

Type for 30 seconds. When the results screen appears — look at the accuracy.

> **Something is off. What do you see?**

---

The tests make the bug concrete. Run this cell:

In [ ]:
repo = workspace / "open_type_faster"
python = repo / "venv" / "bin" / "python"

result = subprocess.run(
    [str(python), "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True, cwd=str(repo)
)
print(result.stdout[-3000:])

3 tests failing, all in `TestAccuracy`:

- `_accuracy(5, 10)` should return `50.0` — returns `0`
- `_accuracy(2, 3)` should return `66.7` — returns `0`
- A session with one error should show `80.0%` — shows `0`

The accuracy calculation always returns 0. Ask the agent to find out why.

---
## Section 4 — Ask the Agent to Fix It

### 🖥️ In the TUI

```
The typing app tests are failing — accuracy always returns 0. Run the tests, find the bug in the source, fix it, and re-run to verify.
```

Watch the loop the agent runs:

1. **Run** `pytest` — see which tests fail and what they expect
2. **Read** the failing test — understand the correct behaviour
3. **Read** `stats.py` — trace the calculation back to the bug
4. **Fix** the minimal change needed
5. **Re-run** to verify

This think → act → observe → repeat loop is what separates an agent from a chatbot.

In [ ]:
# Run after the agent says it's done
result = subprocess.run(
    [str(python), "-m", "pytest", "tests/", "-v"],
    capture_output=True, text=True, cwd=str(repo)
)
print(result.stdout[-2000:])
passed = "failed" not in result.stdout
print("\n" + ("✅ All tests passing" if passed else "❌ Some tests still failing"))

---
## Section 5 — Turn It Into a Skill

What the agent just did was a one-off. A **skill** packages that behaviour so it can be invoked on any Python project — not just this one. Skills live in `~/.openclaw/workspace/skills/` as a folder containing a `SKILL.md` file with YAML frontmatter and step-by-step instructions. OpenClaw injects them into the system prompt automatically.

### 🖥️ In the TUI

```
Create a skill called pytest-debugger in the skills directory. The SKILL.md must define these steps: 1) Read the tests/ folder to understand what is being tested. 2) Run pytest with verbose output and short tracebacks. 3) For each failing test, read the source file it references. 4) Identify the minimal fix — do not rewrite functions. 5) Apply the fix and re-run pytest to confirm. 6) Report: file changed, line changed, what was wrong, what the fix was.
```

When the agent is done, run the next cell to see what it wrote.

In [ ]:
# SKILL.md can be uppercase or lowercase depending on what the agent wrote
# Search for either
skill_dir = workspace / "skills" / "pytest-debugger"
skill_file = None
for candidate in ["SKILL.md", "skill.md"]:
    if (skill_dir / candidate).exists():
        skill_file = skill_dir / candidate
        break

if skill_file:
    print("=" * 60)
    print(f"skills/pytest-debugger/{skill_file.name}")
    print("=" * 60)
    print(skill_file.read_text())
else:
    print(f"Skill not found at {skill_dir}")
    print("Check what the agent created:")
    for p in workspace.rglob("*SKILL*"):
        print(f"  {p}")

That file is all it takes. The skill lives in the workspace alongside `IDENTITY.md` and `SOUL.md` — OpenClaw injects it into every session automatically. Any time you ask the agent to debug a Python project, it has these exact steps to follow.

Skills are just markdown files. You can read them, edit them, and version-control them.

---
## Section 6 — Use the Skill

The skill is now in the workspace. OpenClaw injects it into the agent's system prompt automatically — there's no special command to invoke it. You just describe what you want in natural language, and the agent uses the skill instructions as its guide.

### 🖥️ In the TUI

```
Use the pytest-debugger skill to find and fix the failing tests in ~/.openclaw/workspace/open_type_faster
```

The agent will read the `SKILL.md` steps and follow them — without any further guidance from you.

---
## Section 7 — Morning Brief

You've built an agent that fixes bugs on demand. Now let's make it work *for you* — automatically, every morning, without you lifting a finger.

As an AI/ML developer you need to track fast-moving repos (SGLang, vLLM, Transformers, ROCm, OpenClaw) and keep up with daily AI hardware news. Instead of writing scripts or configuring schedulers, you're going to tell the agent what your ideal morning looks like — and let it figure out the rest.

What it will do autonomously:
- Schedule itself to run every morning at 8 AM
- Save your filtering preferences to long-term memory (`MEMORY.md`)
- Check GitHub repos for relevant PRs and changes
- Search the web for AI hardware news
- Write the brief to your workspace

---

### Part 1 — Delegate (don't mention cron jobs)

Notice the prompt below is purely conversational — no mention of cron, no memory commands. The agent infers what infrastructure it needs.

### 🖥️ In the TUI

```
I need to wake up to a personalized tech brief every morning at 8 AM. Check sgl-project/sglang, vllm-project/vllm, huggingface/transformers, ROCm/ROCm, and openclaw/openclaw. I only care about performance updates, GPU features, and breaking changes — skip CI/infrastructure noise and docs-only PRs. Also search the web and add a summary of the latest AI hardware news.
```

Come back here once the agent confirms it has set up the schedule.

---

### Part 2 — Peek under the hood

The agent didn't just answer — it wrote things to disk. Run the next two cells to verify.

In [ ]:
# Verify the agent scheduled itself
result = subprocess.run(["openclaw", "cron", "list"], capture_output=True, text=True)
print(result.stdout or result.stderr)
# Note: a delivery error is fine here — it means the channel isn't configured, but the schedule is set

In [ ]:
# See what the agent wrote to long-term memory
print("=" * 60)
print("MEMORY.md  (your filtering preferences live here)")
print("=" * 60)
memory_path = workspace / "MEMORY.md"
if memory_path.exists():
    print(memory_path.read_text())
else:
    print("MEMORY.md not found — the agent may not have updated it yet")

Your filtering rules — "skip CI noise", "only GPU features" — are now in `MEMORY.md`. Every future session reads this file at startup. You can update your preferences anytime just by telling the agent in plain language ("actually, include HuggingFace trending models too") and it will rewrite its own memory.

Now force a run — don't wait until 8 AM.

### 🖥️ In the TUI

```
Run my morning brief right now and show me the output.
```

---
## Section 8 — ClawHub

Everything built so far is local to your workspace. ClawHub makes it shareable.

```
Local skill  →  openclaw skill publish  →  ClawHub registry
Anyone       →  openclaw skill install pytest-debugger  →  their workspace
```

The skill travels with its context — whoever installs it gets the same behaviour, not a blank agent.

---

## What Happened Here

| Step | What the agent did | Where it wrote it |
|---|---|---|
| Onboarding | Learned your name, style, preferences | `IDENTITY.md`, `SOUL.md` |
| Clone | Pulled a real repo and set it up | Shell tools |
| Debug | Ran tests, traced a bug, applied a fix | Edited `stats.py` directly |
| Skill | Packaged the debugging approach | `skills/pytest-debugger/SKILL.md` |
| Morning brief | Scheduled itself, saved memory, fetched live data | `MEMORY.md`, cron, web tools |

Every one of those outputs is a plain file you can read, edit, and share.

---

## 🏆 AMD Local Agent Challenge

You've built a working agent. Now use it.

**Download the challenge files first:**
```bash
cd ~/.openclaw/workspace && unzip ~/amd-agent-challenge.zip
```

**Pick a track:**

| Track | What you build | Input |
|---|---|---|
| 🧾 **Track A** — AMD Research Agent | Extract insights from whitepapers, filter noise, produce a structured report | Folder of AMD architecture PDFs |
| 🗂️ **Track B** — Folder Cleanup Agent | Reason about 51 chaotic files, detect duplicates, reorganise into a clean structure | Messy dev downloads folder |
| 🐛 **Track C** — Skill Reuse | Prove the `pytest-debugger` skill works on a brand new codebase it has never seen | Fresh buggy Python project |

---

### 🧾 Track A — AMD Research Agent

**The folder:** `~/.openclaw/workspace/track-a-pdfs/`

A mix of real AMD architecture whitepapers (CDNA 1→4, RDNA4, Chiplet, Advancing AI 2025) plus a couple of non-AMD papers mixed in. Your agent needs to read them, extract insights, and produce a structured report.

### 🖥️ In the TUI
```
Analyse all PDFs in ~/.openclaw/workspace/track-a-pdfs/, identify which ones are AMD-specific, extract the top insights from each, and write a summary report to ~/amd-research-report.md
```

**What a winning output looks like:**
- Correctly filters out the non-AMD decoy papers
- Extracts key metrics, architecture highlights, and trends per document
- Produces a cross-document comparison (e.g. CDNA architecture evolution CDNA1 → CDNA4)
- Structured markdown report with sections, not just a wall of text

---

### 🗂️ Track B — Folder Cleanup Agent

**The folder:** `~/.openclaw/workspace/track-b-downloads/`

51 files: duplicates with `(1)` `(2)` suffixes, UUID-named mystery files, installers, notebooks, logs, trace files, and noise. Your agent needs to reason about what belongs where and reorganise it.

### 🖥️ In the TUI
```
Organise the messy folder at ~/.openclaw/workspace/track-b-downloads/ into a clean structure, detect duplicates, and write a summary of what was done to ~/cleanup-report.md
```

**What a winning output looks like:**
```
track-b-downloads/
  AMD/
    Drivers/
    Notebooks/
    Docs/
  Traces/
  Images/
  Misc/
cleanup-report.md  ← "Organised 51 files, removed 8 duplicates, created 5 categories"
```

---

### 🐛 Track C — Skill Reuse

You built `pytest-debugger` on the typing app. Now prove it generalises.

▶ **Run the cell below** to generate a fresh buggy Python project, then point the skill at it.

### 🖥️ In the TUI
```
Use the pytest-debugger skill to find and fix the bugs in ~/.openclaw/workspace/my-buggy-project
```

**What a winning output looks like:**
- Agent follows the exact `SKILL.md` steps on a codebase it has never seen
- Reports: file changed, line changed, before → after
- All tests pass after the fix
- Bonus: bug report written to `bug_report.md`

---

> "This agent is not a chatbot. It's acting directly on the machine — on AMD hardware, with no cloud, no accounts."

In [ ]:
# Track C — generates the buggy project in your workspace
project = workspace / "my-buggy-project"
(project / "tests").mkdir(parents=True, exist_ok=True)

(project / "calculator.py").write_text('''\
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a // b          # BUG: integer division instead of true division

def is_even(n):
    return n % 2 == 1      # BUG: returns True for odd numbers, not even
''')

(project / "tests" / "__init__.py").write_text("")
(project / "tests" / "test_calculator.py").write_text('''\
import pytest
from calculator import add, subtract, multiply, divide, is_even

def test_add():           assert add(2, 3) == 5
def test_subtract():      assert subtract(10, 4) == 6
def test_multiply():      assert multiply(3, 4) == 12
def test_divide_exact():  assert divide(10, 2) == 5.0
def test_divide_float():  assert divide(7, 2) == 3.5   # catches the // bug
def test_is_even_true():  assert is_even(4) is True     # catches the % bug
def test_is_even_false(): assert is_even(3) is False
''')

print(f"✅ Project ready at {project}")
print("   2 bugs planted — the agent has to find them both.")